
### 10MIN Buckets (Real-Time Dashboard Inputs)
1. Load the raw 10-minute CSVs.

2. Standardize time zones to Manila Time (PHT).

3. Keep the required fields: PM1, PM2.5, PM10, CO2, TVOC, Temperature, Humidity.

4. Filter out impossible readings per field (bounds check).

5. Drop stuck sensor values (flatline) for key fields.

6. Remove localized noise spikes (rolling z-score) for PM columns.

7. Aggregate to 1-hour blocks for modeling consistency (optional).


### 1HR Buckets (72-Hour Forecast Inputs)
1. Load the historical 1-hour CSVs.

2. Standardize time zones to Manila Time (PHT).

3. Keep the required fields: PM1, PM2.5, PM10, Temperature, Humidity.

4. Filter out impossible readings and stuck values.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve data folder from workspace root or notebook folder
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("..") / "data"

RAW_DIR = DATA_DIR / "raw"
if not RAW_DIR.exists():
    RAW_DIR = DATA_DIR

OUTPUT_DIR = DATA_DIR / "cleaned"
MIN_DIR = OUTPUT_DIR / "min-buckets"
HOUR_DIR = OUTPUT_DIR / "hour-buckets"

MIN_DIR.mkdir(parents=True, exist_ok=True)
HOUR_DIR.mkdir(parents=True, exist_ok=True)

# Configure bounds (adjust if needed)
BOUNDS = {
    "pm1": (0, 500),
    "pm25": (0, 500),
    "pm10": (0, 500),
    "co2": (300, 5000),
    "tvoc": (0, 5000),
    "temperature_c": (-10, 60),
    "humidity_pct": (0, 100),
}


def find_timestamp_col(columns: list[str]) -> str:
    for candidate in ["UTC Date/Time", "timestamp", "Local Date/Time"]:
        if candidate in columns:
            return candidate
    raise ValueError("No timestamp column found.")


def pick_column(columns: list[str], startswith: str, prefer_contains: list[str]) -> str | None:
    matches = [c for c in columns if str(c).startswith(startswith)]
    if not matches:
        return None
    preferred = [
        c for c in matches
        if any(p in c.lower() for p in prefer_contains)
    ]
    return preferred[0] if preferred else matches[0]


def find_metric_columns(columns: list[str]) -> dict[str, str]:
    lower_cols = [c.lower() for c in columns]
    col_map: dict[str, str] = {}

    pm1 = pick_column(columns, "PM1", ["corr", "correct", " c"])
    pm25 = pick_column(columns, "PM2.5", ["corr", "correct", " c"])
    pm10 = pick_column(columns, "PM10", ["corr", "correct", " c"])
    co2 = None
    for candidate in ["CO2 (ppm) corrected", "CO2 (ppm) raw", "CO2 (ppm)"]:
        if candidate in columns:
            co2 = candidate
            break
    tvoc = None
    for candidate in ["TVOC (ppb)", "TVOC", "TVOC (ppb) corrected"]:
        if candidate in columns:
            tvoc = candidate
            break
    temp = None
    for i, col in enumerate(columns):
        if "temperature" in lower_cols[i]:
            if "corre" in lower_cols[i] or "correct" in lower_cols[i]:
                temp = col
                break
    if temp is None:
        for i, col in enumerate(columns):
            if "temperature" in lower_cols[i]:
                temp = col
                break
    humidity = None
    for i, col in enumerate(columns):
        if "humidity" in lower_cols[i]:
            if "corre" in lower_cols[i] or "correct" in lower_cols[i]:
                humidity = col
                break
    if humidity is None:
        for i, col in enumerate(columns):
            if "humidity" in lower_cols[i]:
                humidity = col
                break

    if pm1:
        col_map["pm1"] = pm1
    if pm25:
        col_map["pm25"] = pm25
    if pm10:
        col_map["pm10"] = pm10
    if co2:
        col_map["co2"] = co2
    if tvoc:
        col_map["tvoc"] = tvoc
    if temp:
        col_map["temperature_c"] = temp
    if humidity:
        col_map["humidity_pct"] = humidity
    return col_map


def apply_bounds(df: pd.DataFrame, col_map: dict[str, str]) -> pd.DataFrame:
    df = df.copy()
    for key, col in col_map.items():
        if key in BOUNDS:
            low, high = BOUNDS[key]
            df = df[(df[col].isna()) | ((df[col] >= low) & (df[col] <= high))]
    return df


def drop_flatlines(df: pd.DataFrame, col: str, max_run: int) -> pd.DataFrame:
    values = df[col].round(2)
    run_id = (values != values.shift()).cumsum()
    run_len = values.groupby(run_id).transform("size")
    return df[run_len < max_run]


def rolling_outlier_filter(df: pd.DataFrame, col: str, window: int, z_max: float) -> pd.DataFrame:
    rolling_median = df[col].rolling(window=window, min_periods=window // 2, center=True).median()
    rolling_std = df[col].rolling(window=window, min_periods=window // 2, center=True).std()
    z = (df[col] - rolling_median).abs() / rolling_std
    df = df.copy()
    df.loc[z > z_max, col] = np.nan
    return df.dropna(subset=[col])


def clean_10min_df(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, str]]:
    ts_col = find_timestamp_col(df.columns.tolist())
    col_map = find_metric_columns(df.columns.tolist())
    if "pm25" not in col_map:
        raise ValueError("No PM2.5 column found.")

    df = df.copy()
    df["timestamp"] = pd.to_datetime(df[ts_col], utc=True, errors="coerce")
    df["timestamp"] = df["timestamp"].dt.tz_convert("Asia/Manila")
    df = df.dropna(subset=["timestamp"]).sort_values("timestamp")

    # Keep only needed columns plus timestamp
    keep_cols = ["timestamp"] + list(col_map.values())
    df = df[keep_cols]

    # Bounds check
    df = apply_bounds(df, col_map)

    # Flatline removal for key fields
    for key in ["pm25", "pm1", "pm10", "temperature_c", "humidity_pct"]:
        if key in col_map:
            df = drop_flatlines(df, col_map[key], max_run=4)

    # Rolling outlier removal for PM columns
    for key in ["pm25", "pm1", "pm10"]:
        if key in col_map:
            df = rolling_outlier_filter(df, col_map[key], window=12, z_max=3)

    return df, col_map


def aggregate_hourly(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["hour"] = df["timestamp"].dt.floor("H")

    counts = df.groupby("hour")["timestamp"].count()
    keep_hours = counts[counts >= 5].index
    df = df[df["hour"].isin(keep_hours)]

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    agg_map = {col: "mean" for col in numeric_cols}

    non_numeric = [
        c for c in df.columns
        if c not in numeric_cols and c not in ["timestamp", "hour"]
    ]
    agg_map.update({col: "first" for col in non_numeric})

    hourly = df.groupby("hour", as_index=False).agg(agg_map)
    hourly = hourly.rename(columns={"hour": "timestamp"})
    return hourly


summary = []
for file_path in sorted(RAW_DIR.glob("*-10MIN.csv")):
    df_raw = pd.read_csv(file_path, encoding="utf-8", encoding_errors="replace")
    df_clean, col_map = clean_10min_df(df_raw)
    df_hourly = aggregate_hourly(df_clean)

    clean_name = file_path.stem + "-clean.csv"
    hourly_name = file_path.stem.replace("-10MIN", "-10MIN-Agg") + ".csv"

    df_clean.to_csv(MIN_DIR / clean_name, index=False)
    df_hourly.to_csv(HOUR_DIR / hourly_name, index=False)

    summary.append(
        (file_path.name, len(df_raw), len(df_clean), len(df_hourly), list(col_map.keys()))
    )

summary_df = pd.DataFrame(
    summary,
    columns=["file", "raw_rows", "clean_10min_rows", "hourly_rows", "fields"],
)
summary_df

C:\Users\Andre Llanes\AppData\Local\Temp\ipykernel_15600\3234743947.py:166: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Andre Llanes\AppData\Local\Temp\ipykernel_15600\3234743947.py:166: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Andre Llanes\AppData\Local\Temp\ipykernel_15600\3234743947.py:166: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Andre Llanes\AppData\Local\Temp\ipykernel_15600\3234743947.py:166: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Andre Llanes\AppData\Local\Temp\ipykernel_15600\3234743947.py:166: FutureWarning: 'H' is deprecated and will be rem

,file,raw_rows,clean_10min_rows,hourly_rows,fields
0,cubao-10MIN.csv,19898,11134,1397,"[pm1, pm25, pm10, co2, tvoc, temperature_c, hu..."
1,lawton-10MIN.csv,18882,10665,1324,"[pm1, pm25, pm10, co2, tvoc, temperature_c, hu..."
2,makati-10MIN.csv,2223,672,66,"[pm1, pm25, pm10, co2, tvoc, temperature_c, hu..."
3,pasay-10MIN.csv,4936,1871,187,"[pm1, pm25, pm10, co2, tvoc, temperature_c, hu..."
4,SDG-10MIN.csv,663,342,36,"[pm1, pm25, pm10, co2, tvoc, temperature_c, hu..."


STEP 2 - CLEAN HISTORICAL 1HR DATA

In [ ]:
def clean_1hr_df(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, str]]:
    ts_col = find_timestamp_col(df.columns.tolist())
    col_map = find_metric_columns(df.columns.tolist())
    required = ["pm1", "pm25", "pm10", "temperature_c", "humidity_pct"]
    missing = [r for r in required if r not in col_map]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.copy()
    df["timestamp"] = pd.to_datetime(df[ts_col], utc=True, errors="coerce")
    df["timestamp"] = df["timestamp"].dt.tz_convert("Asia/Manila")
    df = df.dropna(subset=["timestamp"]).sort_values("timestamp")

    keep_cols = ["timestamp"] + [col_map[k] for k in required]
    df = df[keep_cols]
    col_map = {k: v for k, v in col_map.items() if v in df.columns}

    # Bounds check
    df = apply_bounds(df, col_map)

    # Flatline removal for key fields (>= 3 consecutive hourly readings)
    for key in required:
        df = drop_flatlines(df, col_map[key], max_run=3)

    return df, col_map


summary_1hr = []
for file_path in sorted(RAW_DIR.glob("*-1HR.csv")):
    df_raw = pd.read_csv(file_path, encoding="utf-8", encoding_errors="replace")
    df_clean, col_map = clean_1hr_df(df_raw)

    clean_name = file_path.stem + "-clean.csv"
    df_clean.to_csv(HOUR_DIR / clean_name, index=False)

    summary_1hr.append(
        (file_path.name, len(df_raw), len(df_clean), list(col_map.keys()))
    )

summary_1hr_df = pd.DataFrame(
    summary_1hr,
    columns=["file", "raw_rows", "clean_rows", "fields"],
)
summary_1hr_df

KeyError: 'CO2 (ppm) corrected'

In [ ]:
from pathlib import Path
import pandas as pd

# Resolve data folder (safe to rerun independently)
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("..") / "data"

OUTPUT_DIR = DATA_DIR / "cleaned"
HOUR_DIR = OUTPUT_DIR / "hour-buckets"
MERGED_DIR = OUTPUT_DIR / "merged"
MERGED_DIR.mkdir(parents=True, exist_ok=True)

def base_from_stem(stem: str, suffix: str) -> str:
    if stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem

agg_files = sorted(HOUR_DIR.glob("*-10MIN-Agg.csv"))
hour_files = sorted(HOUR_DIR.glob("*-1HR-clean.csv"))

agg_map = {base_from_stem(p.stem, "-10MIN-Agg"): p for p in agg_files}
hour_map = {base_from_stem(p.stem, "-1HR-clean"): p for p in hour_files}

merged_summary = []
for base in sorted(set(agg_map) | set(hour_map)):
    parts = []
    if base in agg_map:
        parts.append(pd.read_csv(agg_map[base]))
    if base in hour_map:
        parts.append(pd.read_csv(hour_map[base]))
    if not parts:
        continue
    merged = pd.concat(parts, ignore_index=True)
    if "timestamp" in merged.columns:
        merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
        merged = merged.dropna(subset=["timestamp"]).sort_values("timestamp")
        merged = merged.drop_duplicates(subset=["timestamp"], keep="last")
    merged_file = MERGED_DIR / f"{base}-merged.csv"
    merged.to_csv(merged_file, index=False)
    merged_summary.append((base, len(merged), merged_file.name))

merged_summary_df = pd.DataFrame(
    merged_summary,
    columns=["location", "merged_rows", "file"],
)
merged_summary_df